In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for reference (UTC+8) as required by rules
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into DataFrames (kept in memory for later steps)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Helper summarizers
def file_basic_summary(df, id_col='cmdb_id', timestamp_col='timestamp', max_ids=50):
    total_rows = int(df.shape[0])
    unique_cmdb_count = int(df[id_col].nunique()) if id_col in df.columns else 0
    # preserve original order of appearance for sample list
    cmdb_sample = df[id_col].drop_duplicates().iloc[:max_ids].tolist() if id_col in df.columns else []
    min_ts = df[timestamp_col].min() if timestamp_col in df.columns else None
    max_ts = df[timestamp_col].max() if timestamp_col in df.columns else None
    return {
        'total_rows': total_rows,
        'unique_cmdb_id_count': unique_cmdb_count,
        'cmdb_id_sample_up_to_50': cmdb_sample,
        'min_timestamp': int(min_ts) if pd.notnull(min_ts) else None,
        'max_timestamp': int(max_ts) if pd.notnull(max_ts) else None
    }

# Metric specifics: kpi_name counts (limit 50)
metric_basic = file_basic_summary(df_metric, id_col='cmdb_id', timestamp_col='timestamp', max_ids=50)
if 'kpi_name' in df_metric.columns:
    kpi_counts = df_metric.groupby('kpi_name').size().sort_values(ascending=False).reset_index(name='count').head(50)
else:
    kpi_counts = pd.DataFrame(columns=['kpi_name','count'])

metric_summary = dict(metric_basic)
metric_summary.update({
    'unique_kpi_names_up_to_50': kpi_counts['kpi_name'].tolist(),
    'kpi_counts_top_up_to_50': kpi_counts  # DataFrame (compact, top 50)
})

# Trace specifics: trace_name counts (limit 50)
trace_basic = file_basic_summary(df_trace, id_col='cmdb_id', timestamp_col='timestamp', max_ids=50)
if 'trace_name' in df_trace.columns:
    trace_counts = df_trace.groupby('trace_name').size().sort_values(ascending=False).reset_index(name='count').head(50)
else:
    trace_counts = pd.DataFrame(columns=['trace_name','count'])

trace_summary = dict(trace_basic)
trace_summary.update({
    'unique_trace_names_up_to_50': trace_counts['trace_name'].tolist(),
    'trace_counts_top_up_to_50': trace_counts  # DataFrame
})

# Log specifics: log_name counts (limit 50)
log_basic = file_basic_summary(df_log, id_col='cmdb_id', timestamp_col='timestamp', max_ids=50)
if 'log_name' in df_log.columns:
    log_counts = df_log.groupby('log_name').size().sort_values(ascending=False).reset_index(name='count').head(50)
else:
    log_counts = pd.DataFrame(columns=['log_name','count'])

log_summary = dict(log_basic)
log_summary.update({
    'unique_log_names_up_to_50': log_counts['log_name'].tolist(),
    'log_counts_top_up_to_50': log_counts  # DataFrame
})

# Error logs specifics: distinct message count and up to 20 sample messages
err_basic = file_basic_summary(df_err, id_col='cmdb_id', timestamp_col='timestamp', max_ids=50)
if 'message' in df_err.columns:
    distinct_message_count = int(df_err['message'].nunique())
    sample_messages = df_err['message'].drop_duplicates().iloc[:20].tolist()
else:
    distinct_message_count = 0
    sample_messages = []

error_logs_summary = dict(err_basic)
error_logs_summary.update({
    'distinct_message_count': distinct_message_count,
    'sample_messages_up_to_20': sample_messages
})

# Compose final JSON-like summary (compact). DataFrames included for tabular counts (limited rows).
summary = {
    'metric.csv': metric_summary,
    'trace.csv': trace_summary,
    'log.csv': log_summary,
    'error_logs.csv': error_logs_summary
}

# Display the summary (compact). DataFrames inside the dict are intentionally small (top-N).
summary

```
Out[1]:
```
Summary of telemetry file contents (straightforward):

- metric.csv
  - Total rows: 145,092
  - Unique cmdb_id count: 59
  - Sample cmdb_id values (up to 50 shown): adservice, adservice-0, adservice-1, adservice-2, adservice2, adservice2-0, cartservice, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice, currencyservice-0, currencyservice-1, currencyservice-2, currencyservice2-0, emailservice, emailservice-0, emailservice-1, emailservice-2, emailservice2-0, frontend, frontend-0, frontend-1, frontend-2, frontend2-0, node-1, node-2, node-3, node-4, node-5, node-6, paymentservice, paymentservice-0, paymentservice-1, paymentservice-2, paymentservice2-0, productcatalogservice, productcatalogservice-0, productcatalogservice-1, productcatalogservice-2, productcatalogservice2-0, recommendationservice, recommendationservice-0, recommendationservice-1
  - Timestamp range (raw seconds): min = 1647775800, max = 1647777540
  - KPI names: many unique kpi_name values (first 50 samples shown; many are container.* metrics, several for node-5). Top 50 KPI counts shown range approximately from 1,050 occurrences down to about 990 occurrences.

- trace.csv
  - Total rows: 36,900
  - Unique cmdb_id count: 40
  - Sample cmdb_id values (up to 50 shown): adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice-0, currencyservice-1, currencyservice-2, currencyservice2-0, emailservice-0, emailservice-1, emailservice-2, emailservice2-0, frontend-0, frontend-1, frontend-2, frontend2-0, paymentservice-0, paymentservice-1, paymentservice-2, paymentservice2-0, productcatalogservice-0, productcatalogservice-1, productcatalogservice-2, productcatalogservice2-0, recommendationservice-0, recommendationservice-1, recommendationservice-2, recommendationservice2-0, shippingservice-0, shippingservice-1, shippingservice-2, shippingservice2-0
  - Timestamp range (raw seconds): min = 1647775800, max = 1647777540
  - Trace names: many trace_name values (up to 50 shown, e.g., trace.from_frontend-2.duration_mean, trace.self.duration_mean, trace.to_productcatalogservice-1.duration_mean, etc.). Top trace counts shown range roughly from 630 occurrences down to about 180 occurrences.

- log.csv
  - Total rows: 1,814
  - Unique cmdb_id count: 31
  - Sample cmdb_id values (up to 50 shown): adservice-0, adservice-1, adservice-2, cartservice-0, cartservice-1, cartservice-2, checkoutservice-0, checkoutservice-1, checkoutservice-2, currencyservice-0, currencyservice-1, currencyservice-2, emailservice-0, emailservice-1, emailservice-2, frontend-0, frontend-1, frontend-2, paymentservice-0, paymentservice-1, paymentservice-2, productcatalogservice-0, productcatalogservice-1, productcatalogservice-2, recommendationservice-0, recommendationservice-1, recommendationservice-2, redis-cart-0, shippingservice-0, shippingservice-1, shippingservice-2
  - Timestamp range (raw seconds): min = 1647775800, max = 1647777540
  - Log names and counts: two log_name values found — "log.error_count" and "log.row_count" — each with 907 occurrences.

- error_logs.csv
  - Total rows: 2,278
  - Unique cmdb_id count: 3
  - Sample cmdb_id values (up to 50 shown): frontend-1, frontend-0, frontend-2
  - Timestamp range (raw seconds): min = 1647775800, max = 1647777599
  - Distinct message count: 1
  - Sample message (up to 20): "severity: warning, message: failed to retrieve ads"

If you want any of these summaries expanded (e.g., full list of unique KPI/trace/log names or the full counts table), tell me which file and which columns to show (I can display df.head(X) or filtered counts).

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 145092, 'unique_cmdb_id_count': 59, 'cmdb_id_sample_up_to_50': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1'], 'min_timestamp': 1647775800, 'max_timestamp': 1647777540, 'unique_kpi_names_up_to_50': ['container.node-5.container_network_transmit_MB.eth0', 'container.node-5.container_network_transmit_errors.eth0', 'container.node-5.container_network_transmit_packets.eth0', 'container.node-5.container_network_transmit_packets_dropped.eth0', 'container.node-5.container_network_receive_MB.eth0', 'container.node-5.container_network_receive_packets_dropped.eth0', 'container.node-5.container_network_receive_packets.eth0', 'container.node-5.container_network_receive_errors.eth0', 'container.node-5.container_fs_io_current./dev/vda1', 'container.node-5.container_memory_swap', 'container.node-5.container_memory_usage_MB', 'container.node-5.container_memory_working_set_MB', 'container.node-5.container_cpu_load_average_10s', 'container.node-5.container_last_seen', 'container.node-5.container_cpu_usage_seconds', 'container.node-5.container_cpu_user_seconds', 'container.node-5.container_file_descriptors', 'container.node-5.container_fs_inodes./dev/vda1', 'container.node-5.container_cpu_system_seconds', 'container.node-5.container_fs_writes_merged./dev/vda1', 'container.node-5.container_fs_inodes_free./dev/vda1', 'container.node-5.container_threads_max', 'container.node-5.container_ulimits_soft.max_open_files', 'container.node-5.container_sockets', 'container.node-5.container_spec_cpu_period', 'container.node-5.container_spec_cpu_quota', 'container.node-5.container_spec_cpu_shares', 'container.node-5.container_spec_memory_limit_MB', 'container.node-5.container_spec_memory_reservation_limit_MB', 'container.node-5.container_threads', 'container.node-5.container_tasks_state.uninterruptible', 'container.node-5.container_tasks_state.iowaiting', 'container.node-5.container_tasks_state.running', 'container.node-5.container_tasks_state.stopped', 'container.node-5.container_fs_writes./dev/vda1', 'container.node-5.container_memory_failures.hierarchy.pgfault', 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'container.node-5.container_memory_mapped_file', 'container.node-5.container_memory_max_usage_MB', 'container.node-5.container_fs_sector_writes./dev/vda1', 'container.node-5.container_fs_usage_MB./dev/vda1', 'container.node-5.container_fs_write_seconds./dev/vda1', 'container.node-5.container_cpu_cfs_periods', 'container.node-5.container_cpu_cfs_throttled_periods', 'container.node-5.container_cpu_cfs_throttled_seconds', 'container.node-5.container_memory_cache', 'container.node-5.container_memory_failcnt', 'container.node-5.container_memory_failures.container.pgfault', 'container.node-5.container_tasks_state.sleeping', 'container.node-5.container_spec_memory_swap_limit_MB'], 'kpi_counts_top_up_to_50':                                              kpi_name  count
0   container.node-5.container_network_transmit_MB...   1050
1   container.node-5.container_network_transmit_er...   1050
2   container.node-5.container_network_transmit_pa...   1050
3   container.node-5.container_network_transmit_pa...   1050
4   container.node-5.container_network_receive_MB....   1050
..                                                ...    ...
45            container.node-5.container_memory_cache    990
46          container.node-5.container_memory_failcnt    990
47  container.node-5.container_memory_failures.con...    990
48    container.node-5.container_tasks_state.sleeping    990
49  container.node-5.container_spec_memory_swap_li...    990

[50 rows x 2 columns]}, 'trace.csv': {'total_rows': 36900, 'unique_cmdb_id_count': 40, 'cmdb_id_sample_up_to_50': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0'], 'min_timestamp': 1647775800, 'max_timestamp': 1647777540, 'unique_trace_names_up_to_50': ['trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend-1.row_count', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.row_count', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.self.row_count', 'trace.self.error_rate', 'trace.self.duration_p95', 'trace.self.duration_mean', 'trace.to_productcatalogservice-1.row_count', 'trace.to_productcatalogservice-1.error_rate', 'trace.to_productcatalogservice-1.duration_p95', 'trace.to_productcatalogservice-1.duration_mean', 'trace.to_productcatalogservice-0.row_count', 'trace.to_productcatalogservice-0.error_rate', 'trace.to_productcatalogservice-0.duration_p95', 'trace.to_productcatalogservice-0.duration_mean', 'trace.to_productcatalogservice-2.row_count', 'trace.to_productcatalogservice-2.error_rate', 'trace.to_productcatalogservice-2.duration_p95', 'trace.to_productcatalogservice-2.duration_mean', 'trace.from_root.row_count', 'trace.from_root.error_rate', 'trace.from_root.duration_p95', 'trace.from_root.duration_mean', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.to_currencyservice-0.duration_mean', 'trace.to_currencyservice-0.duration_p95'], 'trace_counts_top_up_to_50':                                   trace_name  count
0        trace.from_frontend-2.duration_mean    630
1         trace.from_frontend-2.duration_p95    630
2           trace.from_frontend-2.error_rate    630
3            trace.from_frontend-2.row_count    630
4            trace.from_frontend-1.row_count    630
..                                       ...    ...
45       trace.from_frontend2-0.duration_p95    210
46         trace.from_frontend2-0.error_rate    210
47          trace.from_frontend2-0.row_count    210
48  trace.to_currencyservice-0.duration_mean    180
49   trace.to_currencyservice-0.duration_p95    180

[50 rows x 2 columns]}, 'log.csv': {'total_rows': 1814, 'unique_cmdb_id_count': 31, 'cmdb_id_sample_up_to_50': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'redis-cart-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2'], 'min_timestamp': 1647775800, 'max_timestamp': 1647777540, 'unique_log_names_up_to_50': ['log.error_count', 'log.row_count'], 'log_counts_top_up_to_50':           log_name  count
0  log.error_count    907
1    log.row_count    907}, 'error_logs.csv': {'total_rows': 2278, 'unique_cmdb_id_count': 3, 'cmdb_id_sample_up_to_50': ['frontend-1', 'frontend-0', 'frontend-2'], 'min_timestamp': 1647775800, 'max_timestamp': 1647777599, 'distinct_message_count': 1, 'sample_messages_up_to_20': ['severity: warning, message: failed to retrieve ads']}}

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# Reuse existing df_metric if present in kernel; otherwise load (defensive)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone object (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) across all available rows (store thresholds)
df_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95_threshold'})
)

# 2) Filter metric.csv to the incident time window: March 20, 2022 19:30 to 20:00 (UTC+8)
start_dt = tz.localize(datetime(2022, 3, 20, 19, 30, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 20, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge thresholds into windowed series
df_window = df_window.merge(df_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Identify anomaly rows where value >= P95 (treat NaN thresholds as non-anomalous)
df_window['is_anomaly'] = False
mask_valid_threshold = df_window['p95_threshold'].notnull()
df_window.loc[mask_valid_threshold, 'is_anomaly'] = df_window.loc[mask_valid_threshold, 'value'] >= df_window.loc[mask_valid_threshold, 'p95_threshold']

df_anomalies = df_window[df_window['is_anomaly']].copy()

# Aggregations:
# - anomaly_count and earliest_anomaly_timestamp from anomalies
anomalies_group = (
    df_anomalies
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(anomaly_count=('value', 'size'),
         earliest_anomaly_timestamp=('timestamp', 'min'))
)

# - total_points_in_window and max_value_in_window from full window
window_group = (
    df_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(total_points_in_window=('value', 'size'),
         max_value_in_window=('value', 'max'))
)

# Compose summary for pairs that had >=1 anomaly
df_summary = (
    anomalies_group
    .merge(window_group, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(df_thresholds, on=['cmdb_id', 'kpi_name'], how='left')
)

# severity_ratio = max_value_in_window / p95_threshold (handle zero threshold)
df_summary['p95_threshold'] = df_summary['p95_threshold'].astype(float)
df_summary['severity_ratio'] = df_summary.apply(
    lambda row: np.inf if (row['p95_threshold'] == 0) else (row['max_value_in_window'] / row['p95_threshold']),
    axis=1
)

# Keep requested columns and sort by severity_ratio desc
df_summary = df_summary[
    ['cmdb_id', 'kpi_name', 'p95_threshold', 'anomaly_count', 'earliest_anomaly_timestamp',
     'max_value_in_window', 'severity_ratio', 'total_points_in_window']
]

# Format numeric columns for compactness (round)
df_summary['p95_threshold'] = df_summary['p95_threshold'].round(6)
df_summary['max_value_in_window'] = df_summary['max_value_in_window'].round(6)
df_summary['severity_ratio'] = df_summary['severity_ratio'].replace(np.inf, 1e12)  # placeholder for display; still large
df_summary['severity_ratio'] = df_summary['severity_ratio'].round(6)

df_summary_sorted = df_summary.sort_values('severity_ratio', ascending=False).reset_index(drop=True)

# 4) Return top 20 rows (compact) and 5) count of distinct pairs with anomalies
df_summary_top20 = df_summary_sorted.head(20)
anomalies_distinct_pairs_count = int(df_summary.shape[0])

# Display compact outputs (DataFrame and count)
df_summary_top20, anomalies_distinct_pairs_count

```
Out[2]:
```
Summary of results (straightforward):

- Window used: March 20, 2022 19:30:00 — 20:00:00 (UTC+8).
- Global P95 thresholds were computed per (cmdb_id, kpi_name) and stored in df_thresholds (as requested, computed on full series before filtering).
- Windowed series stored in df_window.

Key findings:
- Number of distinct (cmdb_id, kpi_name) pairs that had >=1 anomaly in the window (value >= global P95): 4,846.
- Top results (by severity ratio, limited to top 20) are dominated by metrics from cmdb_id = shippingservice2-0. Example pattern for the top entries:
  - cmdb_id: shippingservice2-0
  - many kpi_name values (filesystem, CPU load, memory failure metrics tied to container.node-5)
  - P95_threshold = 0.0
  - anomaly_count = 30
  - earliest_anomaly_timestamp = 1647775800
  - max_value_in_window = 0.0
  - total_points_in_window = 30
  - severity_ratio: shown as a very large number in the table because P95 is zero (division-by-zero); these are effectively infinite/undefined.

Important note/interpretation:
- A large fraction of reported "anomalies" arise because the computed global P95 for those (cmdb_id, kpi_name) pairs is exactly 0.0. Since anomaly logic is value >= P95, zero-valued samples equal to a zero P95 are flagged as anomalies and produce an infinite severity_ratio (we displayed a large placeholder for infinity). This can generate many false-positive-looking anomalies.
- Recommended next steps to refine results: focus on pairs with p95_threshold > 0 (to avoid division-by-zero), or use a different rule (e.g., value > P95 and P95 > 0) or examine only metrics of interest.

If you want, I can:
- show the top pairs after excluding p95 == 0, or
- list pairs with the largest finite severity_ratio, or
- display a compact sample of df_thresholds or df_window for inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  p95_threshold  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  total_points_in_window
0   shippingservice2-0  container.node-5.container_fs_writes_merged./d...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
1   shippingservice2-0  container.node-5.container_memory_failures.con...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
2   shippingservice2-0  container.node-5.container_memory_failures.hie...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
3   shippingservice2-0  container.node-5.container_fs_reads_merged./de...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
4   shippingservice2-0  container.node-5.container_fs_sector_reads./de...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
5   shippingservice2-0  container.node-5.container_fs_sector_writes./d...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
6   shippingservice2-0  container.node-5.container_fs_write_seconds./d...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
7   shippingservice2-0      container.node-5.container_fs_writes./dev/vda            0.0             30                  1647775800                  0.0    1.000000e+12                      30
8   shippingservice2-0     container.node-5.container_fs_writes./dev/vda1            0.0             30                  1647775800                  0.0    1.000000e+12                      30
9   shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda            0.0             30                  1647775800                  0.0    1.000000e+12                      30
10  shippingservice2-0  container.node-5.container_fs_io_current./dev/...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
11  shippingservice2-0  container.node-5.container_fs_io_time_seconds....            0.0             30                  1647775800                  0.0    1.000000e+12                      30
12  shippingservice2-0  container.node-5.container_fs_io_time_weighted...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
13  shippingservice2-0  container.node-5.container_fs_read_seconds./de...            0.0             30                  1647775800                  0.0    1.000000e+12                      30
14  shippingservice2-0       container.node-5.container_fs_reads./dev/vda            0.0             30                  1647775800                  0.0    1.000000e+12                      30
15  shippingservice2-0      container.node-5.container_fs_reads./dev/vda1            0.0             30                  1647775800                  0.0    1.000000e+12                      30
16  shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda            0.0             30                  1647775800                  0.0    1.000000e+12                      30
17  shippingservice2-0    container.node-5.container_cpu_load_average_10s            0.0             30                  1647775800                  0.0    1.000000e+12                      30
18  shippingservice2-0     container.node-5.container_fs_inodes./dev/vda1            0.0             30                  1647775800                  0.0    1.000000e+12                      30
19  shippingservice2-0  container.node-5.container_fs_inodes_free./dev...            0.0             30                  1647775800                  0.0    1.000000e+12                      30, 4846)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_summary from kernel (computed earlier). If not present, raise informative error.
try:
    df_summary
except NameError:
    raise RuntimeError("df_summary not found in kernel. Please run the previous step that computed df_summary.")

# 1) Filter rows to only those with p95_threshold > 0 and finite severity_ratio
# Recompute severity_ratio precisely to avoid placeholder infinities from prior steps
df_filtered = df_summary.copy()

# Ensure numeric types
df_filtered['p95_threshold'] = pd.to_numeric(df_filtered['p95_threshold'], errors='coerce')
df_filtered['max_value_in_window'] = pd.to_numeric(df_filtered['max_value_in_window'], errors='coerce')
df_filtered['anomaly_count'] = pd.to_numeric(df_filtered['anomaly_count'], errors='coerce')
df_filtered['total_points_in_window'] = pd.to_numeric(df_filtered['total_points_in_window'], errors='coerce')

# Compute severity_ratio where p95_threshold > 0
mask_positive_p95 = df_filtered['p95_threshold'] > 0
df_filtered.loc[mask_positive_p95, 'severity_ratio'] = df_filtered.loc[mask_positive_p95, 'max_value_in_window'] / df_filtered.loc[mask_positive_p95, 'p95_threshold']

# Remove rows with p95 <= 0, NaN p95, or non-finite severity_ratio
mask_finite = df_filtered['severity_ratio'].replace([np.inf, -np.inf], np.nan).notnull()
df_filtered = df_filtered[mask_positive_p95 & mask_finite].copy()

# 2) Top 20 rows sorted by severity_ratio descending, include requested columns
cols_out = ['cmdb_id', 'kpi_name', 'p95_threshold', 'anomaly_count', 'earliest_anomaly_timestamp',
            'max_value_in_window', 'severity_ratio', 'total_points_in_window']

df_filtered['p95_threshold'] = df_filtered['p95_threshold'].round(6)
df_filtered['max_value_in_window'] = df_filtered['max_value_in_window'].round(6)
df_filtered['severity_ratio'] = df_filtered['severity_ratio'].round(6)

df_filtered_sorted = df_filtered.sort_values('severity_ratio', ascending=False).reset_index(drop=True)
df_filtered_top20 = df_filtered_sorted[cols_out].head(20)

# 3) Aggregate anomalous activity by cmdb_id (on filtered set where p95>0)
df_agg = (
    df_filtered
    .groupby('cmdb_id', as_index=False)
    .agg(number_of_faulty_kpis=('kpi_name', 'nunique'),
         total_anomaly_count=('anomaly_count', 'sum'),
         earliest_anomaly_timestamp=('earliest_anomaly_timestamp', 'min'))
)

df_agg_top10 = df_agg.sort_values('total_anomaly_count', ascending=False).head(10).reset_index(drop=True)

# 4) Count of distinct (cmdb_id, kpi_name) pairs remaining after excluding p95 == 0
filtered_count = int(df_filtered.shape[0])

# Keep results compact for display: df_filtered_top20, df_agg_top10, filtered_count
df_filtered_top20, df_agg_top10, filtered_count

```
Out[3]:
```
Summary of the filtered anomaly results (p95_threshold > 0 only):

1) Overall count
- Distinct (cmdb_id, kpi_name) pairs remaining after excluding p95 == 0: 2,911

2) Top 20 anomalous KPI pairs (sorted by severity_ratio desc, fields shown)
- Example top entries (severity_ratio, with key fields):
  1. cartservice-2 / mesh.destination.checkoutservice.cartservice.istio_request_latency.p99  
     p95_threshold=0.013650, anomaly_count=4, earliest=1647776280, max_value=42.005775, severity_ratio≈3077.346
  2. cartservice-2 / mesh.destination.frontend.cartservice.istio_request_latency.p95  
     p95_threshold=0.139064, anomaly_count=2, earliest=1647776160, max_value=345.396025, severity_ratio≈2483.720
  3. checkoutservice-1 / mesh.source.checkoutservice.cartservice.istio_request_latency.p99  
     p95_threshold=0.016998, anomaly_count=2, earliest=1647776520, max_value=23.506275, severity_ratio≈1382.885
  4. cartservice / app.grpc.mrt  
     p95_threshold=0.001127, anomaly_count=2, earliest=1647776160, max_value=1.053489, severity_ratio≈934.773
  5. frontend-0 / mesh.source.frontend.cartservice.istio_request_latency.p95  
     p95_threshold=0.171604, anomaly_count=2, earliest=1647775920, max_value=134.397725, severity_ratio≈783.185
  6. frontend-0 / mesh.destination.unknown.frontend.istio_request_time.p95  
     p95_threshold=0.146828, anomaly_count=2, earliest=1647776580, max_value=62.309050, severity_ratio≈424.368
  7. adservice2-0 / container.node-6.container_cpu_cfs_throttled_seconds  
     p95_threshold=0.108920, anomaly_count=2, earliest=1647777060, max_value=11.152624, severity_ratio≈102.393
  8. node-3 / system.io.svctm  
     p95_threshold=0.555000, anomaly_count=2, earliest=1647777000, max_value=48.250000, severity_ratio≈86.937
  9. node-3 / system.io.util  
     p95_threshold=0.355000, anomaly_count=2, earliest=1647775860, max_value=28.950000, severity_ratio≈81.549
  10. adservice2-0 / container.node-6.container_cpu_cfs_throttled_periods  
     p95_threshold=2.100000, anomaly_count=2, earliest=1647777060, max_value=150.000000, severity_ratio≈71.429
  (Rows 11–20 continue similarly; the full top-20 table is stored in kernel variable df_filtered_top20.)

3) Aggregate anomalous activity by cmdb_id (top 10 by total_anomaly_count)
- adservice: number_of_faulty_kpis=268, total_anomaly_count=6,731, earliest_anomaly_timestamp=1647775800
- adservice2: number_of_faulty_kpis=261, total_anomaly_count=6,060, earliest_anomaly_timestamp=1647775800
- checkoutservice-2: number_of_faulty_kpis=78, total_anomaly_count=633, earliest_anomaly_timestamp=1647775800
- recommendationservice-0: number_of_faulty_kpis=43, total_anomaly_count=622, earliest_anomaly_timestamp=1647775800
- checkoutservice2-0: number_of_faulty_kpis=78, total_anomaly_count=620, earliest_anomaly_timestamp=1647775800
- recommendationservice2-0: number_of_faulty_kpis=44, total_anomaly_count=604, earliest_anomaly_timestamp=1647775800
- checkoutservice-1: number_of_faulty_kpis=78, total_anomaly_count=582, earliest_anomaly_timestamp=1647775800
- frontend2-0: number_of_faulty_kpis=85, total_anomaly_count=580, earliest_anomaly_timestamp=1647775800
- checkoutservice-0: number_of_faulty_kpis=78, total_anomaly_count=574, earliest_anomaly_timestamp=1647775800
- emailservice2-0: number_of_faulty_kpis=37, total_anomaly_count=568, earliest_anomaly_timestamp=1647775800

Notes / interpretation
- We intentionally excluded p95 == 0 rows to avoid division-by-zero artifacts. The remaining set (2,911 pairs) shows many high severity_ratio cases, especially for mesh/request-latency and some container IO/CPU metrics.
- Several services (adservice and adservice2) show the largest aggregated anomaly counts and many faulty KPIs in this window — these are likely hotspots to investigate further.
- Earliest anomaly timestamps for top cmdb_ids are all at or near 1647775800 (start of the window range).

If you want, I can:
- return the exact top-20 table as JSON-like rows, or
- show the top cmdb_id rows with timestamps converted to human-readable UTC+8 datetimes.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name  p95_threshold  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  total_points_in_window
0             cartservice-2  mesh.destination.checkoutservice.cartservice.i...       0.013650              4                  1647776280            42.005775     3077.346154                      30
1             cartservice-2  mesh.destination.frontend.cartservice.istio_re...       0.139064              2                  1647776160           345.396025     2483.719906                      30
2         checkoutservice-1  mesh.source.checkoutservice.cartservice.istio_...       0.016998              2                  1647776520            23.506275     1382.884751                      30
3               cartservice                                       app.grpc.mrt       0.001127              2                  1647776160             1.053489      934.772848                      30
4                frontend-0  mesh.source.frontend.cartservice.istio_request...       0.171604              2                  1647775920           134.397725      783.185270                      30
5                frontend-0  mesh.destination.unknown.frontend.istio_reques...       0.146828              2                  1647776580            62.309050      424.367627                      30
6              adservice2-0  container.node-6.container_cpu_cfs_throttled_s...       0.108920              2                  1647777060            11.152624      102.392802                      30
7                    node-3                                    system.io.svctm       0.555000              2                  1647777000            48.250000       86.936937                      30
8                    node-3                                     system.io.util       0.355000              2                  1647775860            28.950000       81.549296                      30
9              adservice2-0  container.node-6.container_cpu_cfs_throttled_p...       2.100000              2                  1647777060           150.000000       71.428571                      30
10        checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...       0.349588              2                  1647776520            23.630000       67.593853                      30
11          checkoutservice                                       app.grpc.mrt       0.057703              2                  1647776040             2.845933       49.320365                      30
12                   node-3                                  system.io.w_await       1.064000              2                  1647777000            48.670000       45.742481                      30
13                   node-3                                    system.io.await       1.064000              2                  1647777000            48.670000       45.742481                      30
14                   node-2                                    system.io.rkb_s       1.100000              2                  1647776640            46.000000       41.818182                      30
15        currencyservice-0  container.node-5.container_cpu_cfs_throttled_s...       0.001601              2                  1647776820             0.064803       40.476577                      30
16               adservice2  runtime.java_lang_Memory_ObjectPendingFinaliza...       0.045833              2                  1647776460             1.500000       32.727511                      30
17                 frontend                                       app.http.mrt       0.047493              2                  1647777360             1.066741       22.461015                      30
18         paymentservice-2  container.node-5.container_network_receive_MB....       0.025836              2                  1647775800             0.550574       21.310342                      30
19  recommendationservice-2      container.node-5.container_fs_writes./dev/vda       1.500000              4                  1647776520            31.500000       21.000000                      30,                     cmdb_id  number_of_faulty_kpis  total_anomaly_count  earliest_anomaly_timestamp
0                 adservice                    268                 6731                  1647775800
1                adservice2                    261                 6060                  1647775800
2         checkoutservice-2                     78                  633                  1647775800
3   recommendationservice-0                     43                  622                  1647775800
4        checkoutservice2-0                     78                  620                  1647775800
5  recommendationservice2-0                     44                  604                  1647775800
6         checkoutservice-1                     78                  582                  1647775800
7               frontend2-0                     85                  580                  1647775800
8         checkoutservice-0                     78                  574                  1647775800
9           emailservice2-0                     37                  568                  1647775800, 2911)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# Reuse df_trace if present; otherwise load (defensive)
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone object (UTC+8)
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 thresholds per (cmdb_id, trace_name) across full series (store)
df_trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95_threshold'})
)

# 2) Filter trace.csv to the incident window: March 20, 2022 19:30:00 — 20:00:00 (UTC+8)
start_dt = tz.localize(datetime(2022, 3, 20, 19, 30, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 20, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds, filter p95>0 and value >= p95, then aggregate per (cmdb_id, trace_name)
df_trace_window = df_trace_window.merge(df_trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Ensure numeric columns
df_trace_window['value'] = pd.to_numeric(df_trace_window['value'], errors='coerce')
df_trace_window['p95_threshold'] = pd.to_numeric(df_trace_window['p95_threshold'], errors='coerce')

# Identify anomaly rows: p95_threshold > 0 and value >= p95_threshold
mask_valid = df_trace_window['p95_threshold'].notnull() & (df_trace_window['p95_threshold'] > 0)
mask_anom = mask_valid & (df_trace_window['value'] >= df_trace_window['p95_threshold'])
df_trace_anomalies = df_trace_window[mask_anom].copy()

# Aggregations per (cmdb_id, trace_name)
df_trace_summary = (
    df_trace_anomalies
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(anomaly_count=('value', 'size'),
         earliest_anomaly_timestamp=('timestamp', 'min'),
         max_value_in_window=('value', 'max'))
)

# total points in window from full window (including non-anomalous)
df_trace_points = (
    df_trace_window
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(total_points_in_window=('value', 'size'))
)

# Compose final summary and compute severity_ratio
df_trace_summary = (
    df_trace_summary
    .merge(df_trace_points, on=['cmdb_id', 'trace_name'], how='left')
    .merge(df_trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
)

df_trace_summary['severity_ratio'] = df_trace_summary['max_value_in_window'] / df_trace_summary['p95_threshold']

# Remove any non-finite ratios (shouldn't exist because p95>0) and NaNs
df_trace_summary = df_trace_summary.replace([np.inf, -np.inf], np.nan).dropna(subset=['severity_ratio', 'p95_threshold'])

# Round numeric columns for compactness
df_trace_summary['p95_threshold'] = df_trace_summary['p95_threshold'].round(6)
df_trace_summary['max_value_in_window'] = df_trace_summary['max_value_in_window'].round(6)
df_trace_summary['severity_ratio'] = df_trace_summary['severity_ratio'].round(6)

# 4) Top 20 anomalous (cmdb_id, trace_name) rows sorted by severity_ratio descending
cols_out = ['cmdb_id', 'trace_name', 'p95_threshold', 'anomaly_count', 'earliest_anomaly_timestamp',
            'max_value_in_window', 'severity_ratio', 'total_points_in_window']

df_trace_summary_sorted = df_trace_summary.sort_values('severity_ratio', ascending=False).reset_index(drop=True)
df_trace_summary_top20 = df_trace_summary_sorted[cols_out].head(20)

# 5) Focused aggregate for components starting with specified prefixes
prefixes = ('frontend', 'adservice', 'cartservice', 'checkoutservice')

mask_prefix = df_trace_summary['cmdb_id'].str.startswith(prefixes)
df_trace_focus = df_trace_summary[mask_prefix].copy()

df_trace_agg_prefix = (
    df_trace_focus
    .groupby('cmdb_id', as_index=False)
    .agg(number_of_faulty_trace_names=('trace_name', 'nunique'),
         total_anomaly_count=('anomaly_count', 'sum'),
         earliest_anomaly_timestamp=('earliest_anomaly_timestamp', 'min'))
)

df_trace_agg_prefix_top20 = df_trace_agg_prefix.sort_values('total_anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Count of distinct (cmdb_id, trace_name) anomalous pairs after p95>0 filtering
trace_anomalous_pairs_count = int(df_trace_summary.shape[0])

# Store key intermediate variables in kernel (already set): df_trace_thresholds, df_trace_window, df_trace_anomalies, df_trace_summary
# Return compact outputs (top tables and counts)
df_trace_summary_top20, df_trace_agg_prefix_top20, trace_anomalous_pairs_count

```
Out[4]:
```
Summary of trace analysis (straightforward):

- Actions performed and storage
  - Global P95 thresholds were computed per (cmdb_id, trace_name) across the full trace series and stored in df_trace_thresholds.
  - Trace data for March 20, 2022 19:30:00 — 20:00:00 (UTC+8) was extracted into df_trace_window.
  - Anomalies (value >= p95 and p95>0) were aggregated into df_trace_summary and individual anomaly rows in df_trace_anomalies.

- Overall count
  - Distinct anomalous (cmdb_id, trace_name) pairs after p95>0 filtering: 881.

- Top anomalous trace pairs (top 6 examples from the top-20 sorted by severity_ratio)
  1. cartservice-2 / trace.from_frontend-0.duration_mean  
     p95_threshold=0.000092, anomaly_count=2, earliest=1647777480, max_value=6.011611, severity_ratio≈65060.725, total_points_in_window=30
  2. frontend-0 / trace.to_cartservice-2.duration_mean  
     p95_threshold=0.000092, anomaly_count=2, earliest=1647777480, max_value=6.011611, severity_ratio≈65060.725, total_points_in_window=30
  3. checkoutservice-1 / trace.to_cartservice-2.duration_mean  
     p95_threshold=0.000367, anomaly_count=2, earliest=1647777420, max_value=20.158000, severity_ratio≈54948.889, total_points_in_window=30
  4. cartservice-2 / trace.from_checkoutservice-1.duration_mean  
     p95_threshold=0.000367, anomaly_count=2, earliest=1647777420, max_value=20.158000, severity_ratio≈54948.889, total_points_in_window=30
  5. cartservice-2 / trace.from_frontend-2.duration_mean  
     p95_threshold=0.000086, anomaly_count=2, earliest=1647777480, max_value=4.629500, severity_ratio≈54051.372, total_points_in_window=30
  6. frontend-2 / trace.to_cartservice-2.duration_mean  
     p95_threshold=0.000086, anomaly_count=2, earliest=1647777480, max_value=4.629500, severity_ratio≈54051.372, total_points_in_window=30

  (Full top-20 is stored in kernel variable df_trace_summary_top20.)

- Focused aggregate for specified components (cmdb_id starting with frontend, adservice, cartservice, checkoutservice)
  - Top rows by total_anomaly_count (showing number_of_faulty_trace_names, total_anomaly_count, earliest timestamp):
    1. checkoutservice-2: 63 faulty trace_names, total_anomaly_count=214, earliest=1647775800
    2. checkoutservice-1: 65 faulty trace_names, total_anomaly_count=202, earliest=1647775800
    3. frontend-1: 67 faulty trace_names, total_anomaly_count=194, earliest=1647775800
    4. checkoutservice-0: 63 faulty trace_names, total_anomaly_count=189, earliest=1647775800
    5. frontend-2: 68 faulty trace_names, total_anomaly_count=176, earliest=1647775800
    6. frontend-0: 68 faulty trace_names, total_anomaly_count=174, earliest=1647775800
    (Up to 20 rows available in df_trace_agg_prefix_top20.)

- Interpretation / notes
  - Many very large severity_ratio values are caused by extremely small p95_thresholds (tiny baselines) — they indicate large relative increases but you should also inspect absolute values and data quality (very small denominators can exaggerate ratios).
  - Earliest anomaly timestamps for top aggregates are at or near 1647775800 (start of the window).
  - Stored kernel variables for follow-up: df_trace_thresholds, df_trace_window, df_trace_anomalies, df_trace_summary, df_trace_summary_top20, df_trace_agg_prefix_top20.

If you want, I can (pick one):
- return the full top-20 table as JSON-like rows, or
- convert earliest timestamps to readable UTC+8 datetimes, or
- re-run the ranking after adding a minimum absolute-value threshold to avoid tiny-p95 artifacts.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                  trace_name  p95_threshold  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  total_points_in_window
0       cartservice-2         trace.from_frontend-0.duration_mean       0.000092              2                  1647777480             6.011611    65060.725108                      30
1          frontend-0        trace.to_cartservice-2.duration_mean       0.000092              2                  1647777480             6.011611    65060.725108                      30
2   checkoutservice-1        trace.to_cartservice-2.duration_mean       0.000367              2                  1647777420            20.158000    54948.889192                      30
3       cartservice-2  trace.from_checkoutservice-1.duration_mean       0.000367              2                  1647777420            20.158000    54948.889192                      30
4       cartservice-2         trace.from_frontend-2.duration_mean       0.000086              2                  1647777480             4.629500    54051.371862                      30
5          frontend-2        trace.to_cartservice-2.duration_mean       0.000086              2                  1647777480             4.629500    54051.371862                      30
6          frontend-2         trace.to_cartservice-2.duration_p95       0.000595              2                  1647777480            19.150500    32185.714286                      30
7       cartservice-2          trace.from_frontend-2.duration_p95       0.000595              2                  1647777480            19.150500    32185.714286                      30
8       cartservice-2          trace.from_frontend-1.duration_p95       0.000797              2                  1647777480            20.596100    25825.830721                      30
9          frontend-1         trace.to_cartservice-2.duration_p95       0.000797              2                  1647777480            20.596100    25825.830721                      30
10         frontend-0         trace.to_cartservice-2.duration_p95       0.001000              3                  1647777000            21.457700    21457.700000                      30
11      cartservice-2          trace.from_frontend-0.duration_p95       0.001000              3                  1647777000            21.457700    21457.700000                      30
12      cartservice-2   trace.from_checkoutservice-1.duration_p95       0.000990              2                  1647777420            20.158000    20361.616162                      30
13  checkoutservice-1         trace.to_cartservice-2.duration_p95       0.000990              2                  1647777420            20.158000    20361.616162                      30
14         frontend-1        trace.to_cartservice-2.duration_mean       0.000832              2                  1647776100             5.734800     6889.476213                      30
15      cartservice-2         trace.from_frontend-1.duration_mean       0.000832              2                  1647776100             5.734800     6889.476213                      30
16      cartservice-2                    trace.self.duration_mean       0.000186              2                  1647776100             0.069953      375.990325                      30
17         frontend-2    trace.to_checkoutservice-1.duration_mean       0.071863              2                  1647776460            23.199168      322.824706                      30
18  checkoutservice-1         trace.from_frontend-2.duration_mean       0.071863              2                  1647776460            23.199168      322.824706                      30
19         frontend-2    trace.to_checkoutservice-0.duration_mean       0.080533              2                  1647776280            18.533147      230.130803                      30,                cmdb_id  number_of_faulty_trace_names  total_anomaly_count  earliest_anomaly_timestamp
0    checkoutservice-2                            63                  214                  1647775800
1    checkoutservice-1                            65                  202                  1647775800
2           frontend-1                            67                  194                  1647775800
3    checkoutservice-0                            63                  189                  1647775800
4           frontend-2                            68                  176                  1647775800
5           frontend-0                            68                  174                  1647775800
6          frontend2-0                            27                   83                  1647775800
7   checkoutservice2-0                            27                   57                  1647775800
8       cartservice2-0                             8                   45                  1647775800
9        cartservice-2                            17                   38                  1647775980
10         adservice-2                             9                   35                  1647775800
11         adservice-1                             9                   31                  1647775860
12       cartservice-0                            12                   30                  1647775920
13       cartservice-1                            12                   28                  1647775920
14         adservice-0                             9                   26                  1647775800
15        adservice2-0                             3                    6                  1647775980, 881)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables where available
# df_trace_summary (aggregated anomalies), df_trace_anomalies, df_trace_thresholds, df_err (error logs) should exist from prior steps.
try:
    df_trace_summary
except NameError:
    raise RuntimeError("df_trace_summary not found in kernel. Please run the previous trace-anomaly computation step first.")

try:
    df_err
except NameError:
    # fallback load if somehow missing
    df_err = pd.read_csv('error_logs.csv')

# Ensure timezone (UTC+8) and incident window timestamps exist; recompute if needed
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(pd.to_datetime("2022-03-20 19:30:00"))
end_dt   = tz.localize(pd.to_datetime("2022-03-20 20:00:00"))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# 1) Filter trace anomalies for 'adservice' in cmdb_id OR trace_name
mask_adservice = (
    df_trace_summary['cmdb_id'].str.contains('adservice', na=False) |
    df_trace_summary['trace_name'].str.contains('adservice', na=False)
)
df_trace_adservice = df_trace_summary[mask_adservice].copy()

# Ensure numeric formatting and finite severity_ratio
for col in ['p95_threshold', 'max_value_in_window', 'severity_ratio']:
    if col in df_trace_adservice.columns:
        df_trace_adservice[col] = pd.to_numeric(df_trace_adservice[col], errors='coerce')

df_trace_adservice = df_trace_adservice.replace([np.inf, -np.inf], np.nan).dropna(subset=['severity_ratio', 'p95_threshold'])

# Top 20 by severity_ratio desc (compact columns)
cols_out = ['cmdb_id', 'trace_name', 'p95_threshold', 'anomaly_count', 'earliest_anomaly_timestamp',
            'max_value_in_window', 'severity_ratio', 'total_points_in_window']

df_trace_adservice['p95_threshold'] = df_trace_adservice['p95_threshold'].round(6)
df_trace_adservice['max_value_in_window'] = df_trace_adservice['max_value_in_window'].round(6)
df_trace_adservice['severity_ratio'] = df_trace_adservice['severity_ratio'].round(6)

df_trace_adservice_top20 = df_trace_adservice.sort_values('severity_ratio', ascending=False)[cols_out].head(20).reset_index(drop=True)

# 2) Aggregate for filtered trace set: distinct_pair_count and overall earliest_anomaly_timestamp
distinct_pair_count = int(df_trace_adservice.shape[0])
earliest_anomaly_ts_adservice = int(df_trace_adservice['earliest_anomaly_timestamp'].min()) if distinct_pair_count > 0 else None
df_trace_adservice_agg = pd.DataFrame([{
    'distinct_pair_count': distinct_pair_count,
    'earliest_anomaly_timestamp': earliest_anomaly_ts_adservice
}])

# 3) Error logs: filter to incident window and aggregate per frontend pod
df_err_window = df_err[(df_err['timestamp'] >= start_ts) & (df_err['timestamp'] <= end_ts)].copy()

# Per-frontend-pod aggregates (cmdb_id startswith 'frontend-')
mask_frontend_pod = df_err_window['cmdb_id'].str.startswith('frontend-', na=False)
df_err_frontend = df_err_window[mask_frontend_pod].copy()

df_err_frontend_agg = (
    df_err_frontend
    .groupby('cmdb_id', as_index=False)
    .agg(message_count=('message', 'size'),
         earliest_message_timestamp=('timestamp', 'min'))
    .sort_values('message_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Overall earliest error log timestamp in the window and up to 5 unique sample messages
overall_earliest_error_ts = int(df_err_window['timestamp'].min()) if not df_err_window.empty else None
err_sample_messages = df_err_window['message'].drop_duplicates().iloc[:5].tolist()

# Store filtered results in kernel variables for follow-up steps
# (variables already assigned: df_trace_adservice, df_trace_adservice_top20, df_trace_adservice_agg,
#  df_err_window, df_err_frontend_agg, overall_earliest_error_ts, err_sample_messages)

# Return compact JSON-like outputs (DataFrames and small scalars/lists)
df_trace_adservice_top20, df_trace_adservice_agg, df_err_frontend_agg, overall_earliest_error_ts, err_sample_messages

```
Out[5]:
```
Summary of results (straightforward):

1) Adservice-related trace anomalies (top examples)
- We filtered df_trace_summary for rows where cmdb_id or trace_name contains "adservice". Top rows (by severity_ratio) include:
  - adservice-1 / trace.from_frontend-1.duration_p95 — p95=0.000021, anomaly_count=5, earliest=1647775920, max=0.000042, severity_ratio≈2.0, total_points=30
  - frontend-1 / trace.to_adservice-1.duration_p95 — p95=0.000021, anomaly_count=5, earliest=1647775920, max=0.000042, severity_ratio≈2.0, total_points=30
  - frontend-2 / trace.to_adservice-2.duration_mean — p95=0.000014, anomaly_count=7, earliest=1647775800, max=0.000021, severity_ratio≈1.5, total_points=30
  - adservice-2 / trace.from_frontend-2.duration_mean — p95=0.000014, anomaly_count=7, earliest=1647775800, max=0.000021, severity_ratio≈1.5, total_points=30
  - (Other similar pairs follow; full filtered top-20 stored in df_trace_adservice_top20.)

2) Aggregate for filtered adservice trace set
- distinct (cmdb_id, trace_name) anomalous pairs: 60
- overall earliest_anomaly_timestamp among them: 1647775800

3) Error logs (incident window March 20, 2022 19:30–20:00 UTC+8)
- Per-frontend-pod aggregates (message_count, earliest_message_timestamp):
  - frontend-0: message_count = 830, earliest_message_timestamp = 1647775800
  - frontend-2: message_count = 762, earliest_message_timestamp = 1647775802
  - frontend-1: message_count = 686, earliest_message_timestamp = 1647775800
- Overall earliest error log timestamp in the window: 1647775800
- Sample error message(s) (up to 5 unique): ["severity: warning, message: failed to retrieve ads"]

Notes:
- Timestamps are raw seconds (UTC+8). If you want, I can convert these to human-readable datetimes or return the exact filtered tables as JSON-like rows. Stored kernel variables for follow-up: df_trace_adservice, df_trace_adservice_top20, df_trace_adservice_agg, df_err_window, df_err_frontend_agg.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                           trace_name  p95_threshold  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  total_points_in_window
0    adservice-1   trace.from_frontend-1.duration_p95       0.000021              5                  1647775920             0.000042        2.000000                      30
1     frontend-1    trace.to_adservice-1.duration_p95       0.000021              5                  1647775920             0.000042        2.000000                      30
2     frontend-2   trace.to_adservice-2.duration_mean       0.000014              7                  1647775800             0.000021        1.500000                      30
3    adservice-2  trace.from_frontend-2.duration_mean       0.000014              7                  1647775800             0.000021        1.500000                      30
4    adservice-1  trace.from_frontend-1.duration_mean       0.000014              5                  1647775920             0.000020        1.428571                      30
5     frontend-1   trace.to_adservice-1.duration_mean       0.000014              5                  1647775920             0.000020        1.428571                      30
6     frontend-1    trace.to_adservice-0.duration_p95       0.000017              2                  1647776040             0.000022        1.329305                      30
7    adservice-0   trace.from_frontend-1.duration_p95       0.000017              2                  1647776040             0.000022        1.329305                      30
8   adservice2-0  trace.from_frontend2-0.duration_p95       0.000035              2                  1647776280             0.000043        1.244573                      30
9    frontend2-0   trace.to_adservice2-0.duration_p95       0.000035              2                  1647776280             0.000043        1.244573                      30
10   adservice-2   trace.from_frontend-2.duration_p95       0.000021              3                  1647776340             0.000026        1.238095                      30
11    frontend-2    trace.to_adservice-2.duration_p95       0.000021              3                  1647776340             0.000026        1.238095                      30
12    frontend-1   trace.to_adservice-2.duration_mean       0.000017              2                  1647775980             0.000021        1.213873                      30
13   adservice-2  trace.from_frontend-1.duration_mean       0.000017              2                  1647775980             0.000021        1.213873                      30
14   adservice-1   trace.from_frontend-2.duration_p95       0.000022              2                  1647776640             0.000025        1.160093                      30
15    frontend-2    trace.to_adservice-1.duration_p95       0.000022              2                  1647776640             0.000025        1.160093                      30
16    frontend-2       trace.to_adservice-2.row_count      30.000000              4                  1647776040            34.000000        1.133333                      30
17   adservice-2      trace.from_frontend-2.row_count      30.000000              4                  1647776040            34.000000        1.133333                      30
18   adservice-0  trace.from_frontend-0.duration_mean       0.000013              2                  1647776100             0.000014        1.115538                      30
19    frontend-0   trace.to_adservice-0.duration_mean       0.000013              2                  1647776100             0.000014        1.115538                      30,    distinct_pair_count  earliest_anomaly_timestamp
0                   60                  1647775800,       cmdb_id  message_count  earliest_message_timestamp
0  frontend-0            830                  1647775800
1  frontend-2            762                  1647775802
2  frontend-1            686                  1647775800, 1647775800, ['severity: warning, message: failed to retrieve ads'])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables
try:
    df_filtered  # this should be the metric anomaly-summary filtered to p95>0 from prior steps
except NameError:
    raise RuntimeError("df_filtered not found in kernel. Please run the metric anomaly filtering step (p95>0) first.")

# Ensure timezone object (UTC+8) available per rules
tz = pytz.timezone('Asia/Shanghai')

# Target cmdb_ids
target_cmdbs = ['adservice-0','adservice-1','adservice-2','adservice2-0']

# Defensive: ensure required columns exist
required_cols = {'cmdb_id','kpi_name','p95_threshold','anomaly_count','earliest_anomaly_timestamp',
                 'max_value_in_window','severity_ratio','total_points_in_window'}
missing = required_cols - set(df_filtered.columns)
if missing:
    raise RuntimeError(f"df_filtered is missing required columns: {missing}")

# Work on a copy for these targets
df_adservice_metrics = df_filtered[df_filtered['cmdb_id'].isin(target_cmdbs)].copy()

# Recompute severity_ratio precisely to avoid any prior placeholders
df_adservice_metrics['p95_threshold'] = pd.to_numeric(df_adservice_metrics['p95_threshold'], errors='coerce')
df_adservice_metrics['max_value_in_window'] = pd.to_numeric(df_adservice_metrics['max_value_in_window'], errors='coerce')
# Avoid division by zero; only keep rows with p95>0 (should already be the case)
mask_pos = df_adservice_metrics['p95_threshold'] > 0
df_adservice_metrics = df_adservice_metrics[mask_pos].copy()
df_adservice_metrics['severity_ratio'] = (df_adservice_metrics['max_value_in_window'] / df_adservice_metrics['p95_threshold']).replace([np.inf, -np.inf], np.nan)

# Round numeric columns for compact display
df_adservice_metrics['p95_threshold'] = df_adservice_metrics['p95_threshold'].round(6)
df_adservice_metrics['max_value_in_window'] = df_adservice_metrics['max_value_in_window'].round(6)
df_adservice_metrics['severity_ratio'] = df_adservice_metrics['severity_ratio'].round(6)

# 1) Top anomalous KPI pairs for these cmdb_ids: top 20 by severity_ratio descending
cols_out = ['cmdb_id','kpi_name','p95_threshold','anomaly_count','earliest_anomaly_timestamp',
            'max_value_in_window','severity_ratio','total_points_in_window']
df_adservice_top20 = df_adservice_metrics.sort_values('severity_ratio', ascending=False)[cols_out].head(20).reset_index(drop=True)

# 2) Aggregate per cmdb_id for these four: number_of_faulty_kpis, total_anomaly_count, earliest_anomaly_timestamp
df_adservice_agg = (
    df_adservice_metrics
    .groupby('cmdb_id', as_index=False)
    .agg(number_of_faulty_kpis=('kpi_name', 'nunique'),
         total_anomaly_count=('anomaly_count', 'sum'),
         earliest_anomaly_timestamp=('earliest_anomaly_timestamp', 'min'))
)

# Ensure order matches requested list (include all four rows even if zero)
df_adservice_agg = pd.DataFrame(target_cmdbs, columns=['cmdb_id']).merge(df_adservice_agg, on='cmdb_id', how='left')
# Fill zeros/NaN for cmdb_ids with no anomalies after filtering
df_adservice_agg['number_of_faulty_kpis'] = df_adservice_agg['number_of_faulty_kpis'].fillna(0).astype(int)
df_adservice_agg['total_anomaly_count'] = df_adservice_agg['total_anomaly_count'].fillna(0).astype(int)
df_adservice_agg['earliest_anomaly_timestamp'] = df_adservice_agg['earliest_anomaly_timestamp'].where(df_adservice_agg['earliest_anomaly_timestamp'].notnull(), None)

# Store filtered results in kernel variables for next steps
df_adservice_metric_filtered = df_adservice_metrics
df_adservice_top20 = df_adservice_top20
df_adservice_agg = df_adservice_agg

# Return compact outputs (DataFrames)
df_adservice_top20, df_adservice_agg

```
Out[6]:
```
Summary (straightforward):

Top anomalous KPI pairs for the specified adservice cmdb_ids (examples from the top results, sorted by severity_ratio):
- adservice2-0 / container.node-6.container_cpu_cfs_throttled_seconds  
  p95=0.108920, anomaly_count=2, earliest=1647777060, max=11.152624, severity_ratio≈102.39, points_in_window=30
- adservice2-0 / container.node-6.container_cpu_cfs_throttled_periods  
  p95=2.100000, anomaly_count=2, earliest=1647777060, max=150.000000, severity_ratio≈71.43, points_in_window=30
- adservice2-0 / container.node-6.container_memory_failures.container.pgfault  
  p95=238.475000, anomaly_count=2, earliest=1647776640, max=3789.000000, severity_ratio≈15.89, points_in_window=30
- adservice2-0 / container.node-6.container_memory_failures.hierarchy.pgfault  
  p95=238.475000, anomaly_count=2, earliest=1647776640, max=3789.000000, severity_ratio≈15.89, points_in_window=30
- adservice2-0 / container.node-6.container_cpu_user_seconds  
  p95=0.376000, anomaly_count=2, earliest=1647777060, max=4.623333, severity_ratio≈12.30, points_in_window=30
- adservice2-0 / container.node-6.container_cpu_usage_seconds  
  p95=0.420802, anomaly_count=2, earliest=1647777060, max=4.671394, severity_ratio≈11.10, points_in_window=30
- adservice-0 / container.node-5.container_network_receive_MB.eth0  
  p95=0.092729, anomaly_count=2, earliest=1647775920, max=0.610487, severity_ratio≈6.58, points_in_window=30
- adservice-1 / mesh.source.adservice.jaeger-collector.istio_request_count  (example mesh metric)  
  p95=0.004675, anomaly_count=3, earliest=1647776100, max=0.007725, severity_ratio≈1.65, points_in_window=30
(Full top-20 for these four cmdb_ids is stored in kernel variable df_adservice_top20.)

Aggregate per cmdb_id (these four):
- adservice-0: number_of_faulty_kpis = 43, total_anomaly_count = 506, earliest_anomaly_timestamp = 1647775800
- adservice-1: number_of_faulty_kpis = 40, total_anomaly_count = 468, earliest_anomaly_timestamp = 1647775800
- adservice-2: number_of_faulty_kpis = 40, total_anomaly_count = 489, earliest_anomaly_timestamp = 1647775800
- adservice2-0: number_of_faulty_kpis = 45, total_anomaly_count = 554, earliest_anomaly_timestamp = 1647775800

Notes/interpretation:
- adservice2-0 shows the highest individual severity ratios (notably CPU throttling and memory-fault related metrics).
- All four cmdb_ids have hundreds of anomaly events and dozens of distinct faulty KPIs in the incident window — adservice2-0 and adservice-0 show the largest totals.
- Earliest anomaly timestamps for these aggregates are at the start of the window (1647775800).
- Filtered results and tables are saved in kernel variables for follow-up: df_adservice_metric_filtered, df_adservice_top20, df_adservice_agg.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  p95_threshold  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  total_points_in_window
0   adservice2-0  container.node-6.container_cpu_cfs_throttled_s...       0.108920              2                  1647777060            11.152624      102.392802                      30
1   adservice2-0  container.node-6.container_cpu_cfs_throttled_p...       2.100000              2                  1647777060           150.000000       71.428571                      30
2   adservice2-0  container.node-6.container_memory_failures.con...     238.475000              2                  1647776640          3789.000000       15.888458                      30
3   adservice2-0  container.node-6.container_memory_failures.hie...     238.475000              2                  1647776640          3789.000000       15.888458                      30
4   adservice2-0        container.node-6.container_cpu_user_seconds       0.376000              2                  1647777060             4.623333       12.296098                      30
5   adservice2-0       container.node-6.container_cpu_usage_seconds       0.420802              2                  1647777060             4.671394       11.101169                      30
6    adservice-0  container.node-5.container_network_receive_MB....       0.092729              2                  1647775920             0.610487        6.583561                      30
7   adservice2-0  container.node-6.container_network_receive_MB....       0.174774              2                  1647777240             0.428985        2.454513                      30
8    adservice-1  mesh.source.adservice.jaeger-collector.istio_r...       0.004675              3                  1647776100             0.007725        1.652406                      30
9    adservice-2  container.node-5.container_network_receive_MB....       0.225668              2                  1647776040             0.349802        1.550074                      30
10   adservice-1  container.node-5.container_network_receive_MB....       0.231731              2                  1647776760             0.355227        1.532928                      30
11  adservice2-0         container.node-6.container_cpu_cfs_periods     188.966667              2                  1647776880           265.333333        1.404128                      30
12   adservice-1  container.node-5.container_memory_failures.con...      34.300000              2                  1647777480            48.000000        1.399417                      30
13   adservice-1  container.node-5.container_memory_failures.hie...      34.300000              2                  1647777480            48.000000        1.399417                      30
14  adservice2-0  mesh.source.adservice2.jaeger-collector.istio_...       0.005216              2                  1647775980             0.007275        1.394747                      30
15   adservice-1      container.node-5.container_cpu_system_seconds       0.025000              2                  1647775920             0.030000        1.200000                      30
16   adservice-2      container.node-5.container_cpu_system_seconds       0.025000              3                  1647775980             0.030000        1.200000                      30
17   adservice-2  mesh.source.adservice.jaeger-collector.istio_r...       0.005225              2                  1647776520             0.006200        1.186603                      30
18   adservice-0  container.node-5.container_network_transmit_MB...       0.890389              2                  1647776700             1.033862        1.161135                      30
19  adservice2-0  container.node-6.container_network_transmit_MB...       1.581135              2                  1647775920             1.805244        1.141739                      30,         cmdb_id  number_of_faulty_kpis  total_anomaly_count  earliest_anomaly_timestamp
0   adservice-0                     43                  506                  1647775800
1   adservice-1                     40                  468                  1647775800
2   adservice-2                     40                  489                  1647775800
3  adservice2-0                     45                  554                  1647775800)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing variables from kernel
try:
    df_adservice_metric_filtered
except NameError:
    raise RuntimeError("df_adservice_metric_filtered not found in kernel. Run previous steps to generate it.")

# Ensure timezone (UTC+8) and incident window timestamps
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 20, 19, 30, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 20, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# Target cmdb ids
target_cmdbs = ['adservice-0','adservice-1','adservice-2','adservice2-0']

# 1) Filter rows where p95_threshold > 0 AND severity_ratio > 1.5 for the target cmdbs
df_ad = df_adservice_metric_filtered.copy()
# Ensure numeric
df_ad['p95_threshold'] = pd.to_numeric(df_ad['p95_threshold'], errors='coerce')
df_ad['severity_ratio'] = pd.to_numeric(df_ad['severity_ratio'], errors='coerce')

mask = (df_ad['cmdb_id'].isin(target_cmdbs)) & (df_ad['p95_threshold'] > 0) & (df_ad['severity_ratio'] > 1.5)
df_ad_severe = df_ad[mask].copy()

# 2) For each cmdb_id find the row with minimum earliest_anomaly_timestamp
# Prepare per-cmdb earliest row
if not df_ad_severe.empty:
    df_ad_severe_sorted = df_ad_severe.sort_values(['cmdb_id','earliest_anomaly_timestamp'], ascending=[True, True])
    df_adservice_severe_earliest = df_ad_severe_sorted.drop_duplicates(subset=['cmdb_id'], keep='first')[[
        'cmdb_id','kpi_name','p95_threshold','earliest_anomaly_timestamp','max_value_in_window',
        'severity_ratio','anomaly_count','total_points_in_window'
    ]].reset_index(drop=True)
else:
    # empty DataFrame with required columns
    df_adservice_severe_earliest = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','p95_threshold','earliest_anomaly_timestamp','max_value_in_window',
        'severity_ratio','anomaly_count','total_points_in_window'
    ])

# Ensure one row per target cmdb_id (include missing ones with NaNs)
df_targets = pd.DataFrame({'cmdb_id': target_cmdbs})
df_adservice_severe_earliest = df_targets.merge(df_adservice_severe_earliest, on='cmdb_id', how='left')

# Round numeric columns for compactness
for col in ['p95_threshold','max_value_in_window','severity_ratio']:
    if col in df_adservice_severe_earliest.columns:
        df_adservice_severe_earliest[col] = pd.to_numeric(df_adservice_severe_earliest[col], errors='coerce').round(6)

# 3) Overall earliest timestamp among these per-pod earliest timestamps
overall_earliest_ts = None
if df_adservice_severe_earliest['earliest_anomaly_timestamp'].notnull().any():
    overall_earliest_ts = int(df_adservice_severe_earliest['earliest_anomaly_timestamp'].dropna().min())

# Frontend error log earliest timestamp in the incident window:
# Prefer df_err_frontend_agg if available in kernel
try:
    df_err_frontend_agg
    frontend_earliest_ts = int(df_err_frontend_agg['earliest_message_timestamp'].min())
except NameError:
    # fallback: compute from df_err (error_logs)
    try:
        df_err
    except NameError:
        df_err = pd.read_csv('error_logs.csv')
    df_err_window = df_err[(df_err['timestamp'] >= start_ts) & (df_err['timestamp'] <= end_ts)]
    df_err_frontend = df_err_window[df_err_window['cmdb_id'].str.startswith('frontend-', na=False)]
    frontend_earliest_ts = int(df_err_frontend['timestamp'].min()) if not df_err_frontend.empty else None

# Store filtered per-pod earliest severe-fault rows in kernel variable
df_adservice_severe_earliest = df_adservice_severe_earliest

# Return compact results: DataFrame (up to 4 rows) and two scalar timestamps
df_adservice_severe_earliest, overall_earliest_ts, frontend_earliest_ts

```
Out[7]:
```
```python
from datetime import datetime
import pytz

# Reuse kernel variables: df_adservice_severe_earliest, overall_earliest_ts, frontend_earliest_ts
tz = pytz.timezone('Asia/Shanghai')

def fmt_ts(ts):
    if ts is None or (pd.isna(ts) if 'pd' in globals() else False):
        return "N/A"
    try:
        ts_int = int(ts)
        dt = datetime.fromtimestamp(ts_int, tz)
        return f"{ts_int} ({dt.strftime('%Y-%m-%d %H:%M:%S %Z')})"
    except Exception:
        return str(ts)

rows = []
for _, r in df_adservice_severe_earliest.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name'] if pd.notnull(r.get('kpi_name')) else "N/A"
    p95 = r['p95_threshold'] if pd.notnull(r.get('p95_threshold')) else "N/A"
    earliest = fmt_ts(r.get('earliest_anomaly_timestamp'))
    maxv = r['max_value_in_window'] if pd.notnull(r.get('max_value_in_window')) else "N/A"
    sev = r['severity_ratio'] if pd.notnull(r.get('severity_ratio')) else "N/A"
    ac = int(r['anomaly_count']) if pd.notnull(r.get('anomaly_count')) else 0
    tp = int(r['total_points_in_window']) if pd.notnull(r.get('total_points_in_window')) else 0
    rows.append(f"- {cmdb}: KPI='{kpi}', p95={p95}, earliest={earliest}, max={maxv}, severity_ratio={sev}, anomaly_count={ac}, total_points={tp}")

overall_earliest_str = fmt_ts(overall_earliest_ts)
frontend_earliest_str = fmt_ts(frontend_earliest_ts)

summary_lines = [
    "Summary of earliest severe KPI faults for adservice pods (filtered p95>0 and severity_ratio>1.5):",
] + rows + [
    "",
    f"Overall earliest severe anomaly among these pods: {overall_earliest_str}",
    f"Frontend error log earliest timestamp in the incident window: {frontend_earliest_str}"
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name  p95_threshold  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  anomaly_count  total_points_in_window
0   adservice-0  container.node-5.container_network_receive_MB....       0.092729                  1647775920             0.610487        6.583561              2                      30
1   adservice-1  mesh.source.adservice.jaeger-collector.istio_r...       0.004675                  1647776100             0.007725        1.652406              3                      30
2   adservice-2  container.node-5.container_network_receive_MB....       0.225668                  1647776040             0.349802        1.550074              2                      30
3  adservice2-0  container.node-6.container_memory_failures.con...     238.475000                  1647776640          3789.000000       15.888458              2                      30, 1647775920, 1647775800)```
```